In [0]:
%pip install tqdm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from glob import glob
import pandas as pd
from tqdm import tqdm

In [0]:
# ── Unity Catalog location (must match the other scripts in this pipeline) ──
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #change
SCHEMA = "tier1_raw" # change
 
# Source table: the flight inventory table generated by Job 1 (1_Flight_table).
flight_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
# Destination table: where this script saves the orthomosaic status inventory.
TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_ortho_table"

In [0]:
# Load the full flight inventory into a local pandas DataFrame so we can
# loop through it row by row.
flight_df = spark.read.table(flight_table).toPandas()
len(flight_df)

6

In [0]:
row_list = []
 
# ── Go through every flight and check its orthomosaic status ───────────────
for idx, row in tqdm(flight_df.iterrows(), total=len(flight_df)):
 
    # 1. We obtain the path of the base folder
    flight_path = os.path.dirname(row['flight_metadata_path'])
 
    # FIX DATABRICKS: Ensure 'os' library can read the path if it uses dbfs:/
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
 
    # 2. Check both possible orthomosaic filenames (RGB.tif or MS.tif).
    # This is how we tell whether Job 2 (PIPELINE_ORTHOMOSAIC) has already
    # generated the orthomosaic for this flight or not.
    possible_names = ["RGB.tif", "MS.tif"]
    ortho_path = None
    ortho_exists = False
 
    for name in possible_names:
        candidate = f"{flight_path}/{name}"
        if os.path.exists(candidate):
            ortho_path = candidate
            ortho_exists = True
            break
 
    # If neither exists, default to RGB.tif as the "expected" path so you can see what's missing
    if ortho_path is None:
        ortho_path = f"{flight_path}/RGB.tif"
 
    # 3. We point to the raw data (using the exact path you provided)
    # This is used purely to count how many source images this flight has,
    # independent of whether the orthomosaic has been built yet.
    raw_data_path = f"{flight_path}/raw_data/rgb"
    image_count = 0
 
    # We count images REGARDLESS of whether the orthomosaic exists yet
    if os.path.exists(raw_data_path):
        files = os.listdir(raw_data_path)
        image_count = len([f for f in files if f.lower().endswith(('.tif', '.jpg', '.jpeg'))])
 
    # 4. We build the dictionary ALWAYS so the DataFrame structure is created
    # (every flight gets a row, whether its orthomosaic exists or not — this
    # is what allows the orchestrator to later identify pending flights).
    row_dict = {
        'site': row['site'],
        'trial': row['trial'],
        'season': row['season'],
        'field':  row['field'],
        'location': row['location'],
        'mission': row['mission'],
        'flight_date': row['flight_date'],
        'ortho_file_path': ortho_path,
        'ortho_exists': ortho_exists, # True if either RGB.tif or MS.tif was found
        'plot_image_count': image_count
    }
 
    row_list.append(row_dict)
 
# 5. We generate the final DataFrame
ortho_df = pd.DataFrame(row_list)
 
print(f"Total flights processed: {len(ortho_df)}")
 
if len(ortho_df) > 0:
    display(ortho_df.head())
else:
    # If this is empty, it usually means Job 1 (flight table generation)
    # hasn't been run yet, since this script depends on that table existing.
    print("It's still empty. Check if you ran the flight_df table before running this.")

100%|██████████| 6/6 [00:01<00:00,  4.30it/s]


Total de ortomosaicos encontrados: 4


site,trial,season,field,location,mission,flight_date,ortho_file_path,plot_image_count
CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_11_24_00_00,/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/Aduthurai/Paddy/2024_11_24_00_00/RGB.tif,0
CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_12_09_00_00,/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/Aduthurai/Paddy/2024_12_09_00_00/RGB.tif,0
CIAT_CALI,Bhavanisagar,Tomato,unknown,unknown,unknown,2025_01_02_00_00,/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/Bhavanisagar/Tomato/2025_01_02_00_00/RGB.tif,0
CIAT_CALI,Kovilpatti,Sorghum,unknown,unknown,unknown,2024_12_02_00_00,/Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/Kovilpatti/Sorghum/2024_12_02_00_00/RGB.tif,0


In [0]:
# just used for a quick preview.)
ortho_df = pd.DataFrame(row_list)
ortho_df.head()

,site,trial,season,field,location,mission,flight_date,ortho_file_path,plot_image_count
0,CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_11_24_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
1,CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_12_09_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
2,CIAT_CALI,Bhavanisagar,Tomato,unknown,unknown,unknown,2025_01_02_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
3,CIAT_CALI,Kovilpatti,Sorghum,unknown,unknown,unknown,2024_12_02_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0


In [0]:
# Quick sanity check: flights whose raw_data folder had 0 images detected.
# Useful for spotting flights with missing/misplaced source images.
ms = ortho_df[ortho_df['plot_image_count'] == 0]
ms.head()

,site,trial,season,field,location,mission,flight_date,ortho_file_path,plot_image_count
0,CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_11_24_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
1,CIAT_CALI,Aduthurai,Paddy,unknown,unknown,unknown,2024_12_09_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
2,CIAT_CALI,Bhavanisagar,Tomato,unknown,unknown,unknown,2025_01_02_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0
3,CIAT_CALI,Kovilpatti,Sorghum,unknown,unknown,unknown,2024_12_02_00_00,/Volumes/use1_prod_artemis_catalog_37181949744...,0


In [0]:
len(ortho_df[ortho_df['plot_image_count'] == 0])

4

In [0]:
# Convert the pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(ortho_df)

# You can now see the schema of the new Spark DataFrame
spark_df.printSchema()

root
 |-- site: string (nullable = true)
 |-- trial: string (nullable = true)
 |-- season: string (nullable = true)
 |-- field: string (nullable = true)
 |-- location: string (nullable = true)
 |-- mission: string (nullable = true)
 |-- flight_date: string (nullable = true)
 |-- ortho_file_path: string (nullable = true)
 |-- plot_image_count: long (nullable = true)



In [0]:
# Write (overwrite) the orthomosaic status table. "mergeSchema" allows the
# table's schema to evolve if new columns are added in future runs.
spark_df.write.option("mergeSchema", "true").saveAsTable(TABLE_NAME, mode="overwrite")